# Dimensionality Reduction & Clustering

Package-backed computational workflow for fitted gene-expression trajectories and dynamic-mode clustering. Publication plotting is intentionally kept out of this notebook for now.


In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import chronopack as cp


In [ ]:
# Experiment directories; this notebook runs from k562-tnfa-puromycin.hg38/notebooks.
root_puro = Path("..").resolve()
root_k562a = Path("../../k562-tnfa.hg38.rep").resolve()
root_lps = Path("../../thp1-k562-lps.hg38.rep").resolve()

SELF_CONTAINED_OUTDIR = root_puro / "self_contained_outputs" / "global_analysis_extended"
SELF_CONTAINED_OUTDIR.mkdir(parents=True, exist_ok=True)

{
    "root_puro": str(root_puro),
    "root_k562a": str(root_k562a),
    "root_lps": str(root_lps),
    "output": str(SELF_CONTAINED_OUTDIR),
}


## Build Canonical Trajectory AnnData
The fitted tradeSeq SCE objects are the inputs; association statistics are read from `rowData(sce)`.

In [ ]:
# Final 8-knot tradeSeq fits written by the TradeSeqFits notebooks.
sce_dict = {
    "K562_puro": root_puro / "intmd" / "sc-K562-1e6-to-tradeSeq.fixed" / "sce.1trajectory.nknots8.rds",
    "K562a": root_k562a / "intmd" / "sc-K562-1e4_to_tradeSeq.fixed" / "sce.1trajectory.nknots8.rds",
    "K562b": root_lps / "intmd" / "sc-K562_to_tradeSeq.fixed" / "sce.1trajectory.nknots8.rds",
    "THP1": root_lps / "intmd" / "sc-THP1_to_tradeSeq.fixed" / "sce.1trajectory.nknots8.rds",
}
traj = cp.tl.align_gene_trajectories(
    sce_dict,
    output_dir=SELF_CONTAINED_OUTDIR / "trajectory_payload",
    time_map={"K562a": 110, "K562_puro": 110, "K562b": 165, "THP1": 165},
    offset_map={"K562a": 0, "K562_puro": 0, "K562b": -68, "THP1": -10},
    scale_map={"THP1": 0.65},
)
cp.tl.compute_gene_trajectory_descriptors(
    traj,
    inplace=True,
    output_dir=SELF_CONTAINED_OUTDIR / "trajectory_payload",
)

traj.write_h5ad(SELF_CONTAINED_OUTDIR / "gene_trajectory_characterization.h5ad")
pd.DataFrame(traj.X, index=traj.obs_names, columns=traj.var_names).to_csv(
    SELF_CONTAINED_OUTDIR / "gene_trajectory_fit.csv"
)
pd.DataFrame({"time": traj.var["time"].to_numpy()}).to_csv(
    SELF_CONTAINED_OUTDIR / "gene_trajectory_time_grid.csv", index=False
)
traj.obs.to_csv(SELF_CONTAINED_OUTDIR / "pc_df.characterized.csv")

{
    "traj_shape": traj.shape,
    "time_grid_start_end": (float(traj.var["time"].iloc[0]), float(traj.var["time"].iloc[-1])),
    "raw_trajectory_size_bytes": traj.uns["chronopack"]["alignment"]["raw_trajectory_size_bytes"],
    "raw_trajectories_embedded": traj.uns["chronopack"]["alignment"]["raw_trajectories_embedded"],
}


## Cluster Gene Trajectories

The notebook explicitly defines induced and repressed clustering populations, computes every candidate K once, visualizes model-selection diagnostics, and then selects labels from the retained candidate assignments.


In [ ]:
LOG2FC_THRESHOLD = 0
PVALUE_THRESHOLD = 0.05
EXCLUDED_CLUSTER_CELL_TYPE = "K562_puro"
ALPHA = 0.1
K_GRID = range(2, 101)

sig = traj[traj.obs["pvalue_1"] < PVALUE_THRESHOLD].copy()
cluster_universe = sig[sig.obs["cell_type"] != EXCLUDED_CLUSTER_CELL_TYPE].copy()

up = cluster_universe[cluster_universe.obs["peak_log2fc"] > LOG2FC_THRESHOLD].copy()
up_clust = cp.tl.cluster_gene_trajectories(up, output_dir=SELF_CONTAINED_OUTDIR / "cluster_candidates" / "induced", k=K_GRID, reference=cluster_universe, alpha=ALPHA)

down = cluster_universe[cluster_universe.obs["peak_log2fc"] < -LOG2FC_THRESHOLD].copy()
down_clust = cp.tl.cluster_gene_trajectories(down, output_dir=SELF_CONTAINED_OUTDIR / "cluster_candidates" / "repressed", k=K_GRID, reference=cluster_universe, alpha=ALPHA)

metrics_by_name = {"induced": up_clust.metrics.copy(), "repressed": down_clust.metrics.copy()}

# Broad modes use a fixed k = 3; fine modes use the k with the highest elbow score.
FORCED_BROAD_K = 3
selected_k = {
    name: {"broad": FORCED_BROAD_K, "fine": int(df.loc[df["elbow_score"].idxmax(), "k" if "k" in df else "K"])}
    for name, df in metrics_by_name.items()
}

metrics_summary = pd.DataFrame({
    name: {
        "k_elbow_for_fine": vals["fine"],
        "k_forced_for_broad": vals["broad"],
    }
    for name, vals in selected_k.items()
}).T

display(metrics_summary)

up_metric_plot = cp.pl.clustering_metrics(up_clust, output=SELF_CONTAINED_OUTDIR / "cluster_metric_selection.induced.png")
down_metric_plot = cp.pl.clustering_metrics(down_clust, output=SELF_CONTAINED_OUTDIR / "cluster_metric_selection.repressed.png")
plt.show()

# Compatibility filenames used by downstream checks and manuscript scripts.
up_clust.metrics.to_csv(SELF_CONTAINED_OUTDIR / "cluster_metrics.induced.csv", index=False)
down_clust.metrics.to_csv(SELF_CONTAINED_OUTDIR / "cluster_metrics.repressed.csv", index=False)

{
    "alpha": ALPHA,
    "significant_rows": sig.n_obs,
    "clustered_rows": cluster_universe.n_obs,
    "excluded_cluster_cell_type": EXCLUDED_CLUSTER_CELL_TYPE,
    "excluded_rows": int((sig.obs["cell_type"] == EXCLUDED_CLUSTER_CELL_TYPE).sum()),
    "induced_rows": up.n_obs,
    "repressed_rows": down.n_obs,
    "forced_broad_k": FORCED_BROAD_K,
    "selected_k": selected_k,
    "metric_plots": [up_metric_plot.output, down_metric_plot.output],
}


### Cluster Metric Selection Plots

The plots below are generated directly by `cp.pl.clustering_metrics()` from the candidate clustering results. The elbow-highlighted K is used for fine clusters; broad clusters use a fixed K = 3.


## Select Candidate Assignments and Name Modes

This final manuscript-specific step combines induced and repressed labels explicitly in the notebook. No ChronoPack finalization helper is used.


In [ ]:
pc_df = traj.obs.copy()
for col in [
    "cluster_broad_induced",
    "cluster_dist_broad_induced",
    "cluster_fine_induced",
    "cluster_dist_fine_induced",
    "cluster_broad_repressed",
    "cluster_dist_broad_repressed",
    "cluster_fine_repressed",
    "cluster_dist_fine_repressed",
]:
    pc_df[col] = np.nan if "dist" in col else "-1"

k_up_broad = selected_k["induced"]["broad"]
k_up_fine = selected_k["induced"]["fine"]
k_down_broad = selected_k["repressed"]["broad"]
k_down_fine = selected_k["repressed"]["fine"]

pc_df.loc[up.obs_names, "cluster_broad_induced"] = up_clust.assignments[k_up_broad].astype(str)
pc_df.loc[up.obs_names, "cluster_dist_broad_induced"] = up_clust.centroid_distances[k_up_broad]
pc_df.loc[up.obs_names, "cluster_fine_induced"] = up_clust.assignments[k_up_fine].astype(str)
pc_df.loc[up.obs_names, "cluster_dist_fine_induced"] = up_clust.centroid_distances[k_up_fine]

pc_df.loc[down.obs_names, "cluster_broad_repressed"] = down_clust.assignments[k_down_broad].astype(str)
pc_df.loc[down.obs_names, "cluster_dist_broad_repressed"] = down_clust.centroid_distances[k_down_broad]
pc_df.loc[down.obs_names, "cluster_fine_repressed"] = down_clust.assignments[k_down_fine].astype(str)
pc_df.loc[down.obs_names, "cluster_dist_fine_repressed"] = down_clust.centroid_distances[k_down_fine]

for col in ["cluster_broad_induced", "cluster_fine_induced", "cluster_broad_repressed", "cluster_fine_repressed"]:
    pc_df[col] = pc_df[col].fillna("-1").astype(str)

rename_induced = {"1": "impulse-up_1", "2": "trend-up", "3": "impulse-up_2"}
rename_repressed = {"1": "trend-down_1", "2": "trend-down_2", "3": "transient-down"}
pc_df["cluster_broad_induced"] = pc_df["cluster_broad_induced"].map(lambda x: rename_induced.get(x, x))
pc_df["cluster_broad_repressed"] = pc_df["cluster_broad_repressed"].map(lambda x: rename_repressed.get(x, x))

pc_df["broad_cluster"] = pc_df["cluster_broad_induced"].where(
    pc_df["cluster_broad_induced"] != "-1",
    pc_df["cluster_broad_repressed"],
)
pc_df["cluster_dist_broad"] = pc_df["cluster_dist_broad_induced"].combine_first(
    pc_df["cluster_dist_broad_repressed"]
)
pc_df["cluster_dist_fine"] = pc_df["cluster_dist_fine_induced"].combine_first(
    pc_df["cluster_dist_fine_repressed"]
)

cell_type_order = ["K562a", "K562b", "K562_puro", "THP1"]
pc_df = pc_df.assign(_cell_type_order=pc_df["cell_type"].map({k: i for i, k in enumerate(cell_type_order)}))
pc_df = pc_df.sort_values("_cell_type_order").drop(columns="_cell_type_order")

for col in pc_df.columns:
    traj.obs[col] = pc_df.loc[traj.obs_names, col]

# Standardized matrices are derived products retained for regression checks and downstream scripts.
fit, fit_z, ddt_z = cp.tl.standardize_gene_trajectories(traj)
fit_z.to_csv(SELF_CONTAINED_OUTDIR / "gene_trajectory_fit_z.csv")
ddt_z.to_csv(SELF_CONTAINED_OUTDIR / "gene_trajectory_fit_ddt_z.csv")
fit.loc[pc_df.index[pc_df["pvalue_1"] < PVALUE_THRESHOLD]].to_csv(
    SELF_CONTAINED_OUTDIR / "gene_trajectory_fit_significant.csv"
)
fit_z.loc[pc_df.index[pc_df["pvalue_1"] < PVALUE_THRESHOLD]].to_csv(
    SELF_CONTAINED_OUTDIR / "gene_trajectory_fit_significant_z.csv"
)
ddt_z.loc[pc_df.index[pc_df["pvalue_1"] < PVALUE_THRESHOLD]].to_csv(
    SELF_CONTAINED_OUTDIR / "gene_trajectory_fit_ddt_significant_z.csv"
)

pc_df.to_csv(SELF_CONTAINED_OUTDIR / "pc_df.new.csv")
traj.write_h5ad(SELF_CONTAINED_OUTDIR / "gene_trajectory_clusters.h5ad")

{
    "traj_shape": traj.shape,
    "pc_df_shape": pc_df.shape,
    "selected_k": selected_k,
    "broad_cluster_counts": pc_df["broad_cluster"].astype(str).value_counts().to_dict(),
}


## Inspect Outputs


In [5]:
display(pc_df.head())
print(pc_df.shape)
print(pc_df["broad_cluster"].value_counts())


(6723, 35)
broad_cluster
trend-down_1      2215
trend-up          1947
transient-down    1027
impulse-up_1       887
impulse-up_2       379
trend-down_2       268
Name: count, dtype: int64


,gene,cell_type,t_onset,t_peak,baseline_exp,peak_exp,peak_dlogexp,peak_dexp,peak_log2fc,auc_norm,owp,owp_ddt,mean_curv,max_curv,mean_turn,max_turn,waldStat_1,meanLogFC,pvalue_1,qval_1,PC1,PC2,PC3,PC4,cluster_broad_induced,cluster_dist_broad_induced,cluster_fine_induced,cluster_dist_fine_induced,cluster_broad_repressed,cluster_dist_broad_repressed,cluster_fine_repressed,cluster_dist_fine_repressed,broad_cluster,cluster_dist_broad,cluster_dist_fine
NFKBIA_K562a,NFKBIA,K562a,13.0,44.0,0.205383,1.669589,1.464206,4.081991,4.240605,0.660197,0.873081,0.254154,3.203454,16.999057,0.013560,0.057604,1967.377559,2.806293,0.0,0.0,-6.760465,-7.896757,1.886260,-1.602042,impulse-up_1,0.652766,1,0.237635,-1,NaN,-1,NaN,impulse-up_1,0.652766,0.237635
CXCL3_K562a,CXCL3,K562a,12.0,40.0,0.073959,0.956644,0.882685,1.526183,4.384173,0.715159,0.922997,0.245469,3.996198,27.634036,0.016469,0.092331,742.323899,2.786376,0.0,0.0,-8.009887,-6.520550,2.490377,-0.687918,impulse-up_1,0.653564,1,0.298007,-1,NaN,-1,NaN,impulse-up_1,0.653564,0.298007
CD69_K562a,CD69,K562a,13.0,41.0,0.031030,0.850327,0.819297,1.308895,5.410430,0.610903,0.810655,0.227155,4.699037,31.936844,0.019736,0.107055,602.364734,3.454632,0.0,0.0,-6.107661,-7.916383,2.446116,-1.639655,impulse-up_1,0.733679,1,0.352253,-1,NaN,-1,NaN,impulse-up_1,0.733679,0.352253
CD83_K562a,CD83,K562a,21.0,97.0,0.025817,0.361206,0.335389,0.408906,4.056148,0.423911,1.000000,0.865431,0.890947,1.705750,0.003982,0.007014,566.998711,2.128998,0.0,0.0,-10.975138,-0.392566,-0.788509,-0.192591,trend-up,0.514878,2,0.395160,-1,NaN,-1,NaN,trend-up,0.514878,0.395160
ARL4A_K562a,ARL4A,K562a,16.0,91.0,1.102629,1.487241,0.384612,1.412797,0.767366,0.558478,0.999284,0.645456,2.466544,11.254000,0.009530,0.037568,466.569370,0.365352,0.0,0.0,-10.762020,-2.443960,-0.546071,-0.291104,impulse-up_1,0.478791,3,0.375831,-1,NaN,-1,NaN,impulse-up_1,0.478791,0.375831


## Optional Reference Validation


In [6]:
import os

if os.environ.get("CHRONOPACK_VALIDATE_REFERENCE") == "1":
    reference_path = root_puro / "intmd" / "global_analysis" / "extended" / "pc_df.new.rds"
    print("Reference validation requested.")
    print("Reference path:", reference_path)
    print("Current rows:", pc_df.shape[0])
    print("Current columns:", pc_df.shape[1])
else:
    print("Set CHRONOPACK_VALIDATE_REFERENCE=1 to run external reference validation.")


Set CHRONOPACK_VALIDATE_REFERENCE=1 to run external reference validation.
